# 🏷️ 17 · Learning with few labels: self-supervision, few-shot, active and open-set learning
### What to do when experts can label only a handful of particles — and new complexes appear in every cell

**BiologyToResearchV3** · Part D — Deep learning on subtomograms (2017–2024) · Module 17 of 26

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/cryoem/tutorial/BiologyToResearchV3/notebooks/17_learning_with_few_labels.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 5–7 min to run everything); a GPU is used automatically if present |
| **Core frameworks** | Python 3, NumPy, SciPy, scikit-learn, PyTorch, Matplotlib |
| **Prerequisites** | Module 15 (3D CNNs, the 15 Å / 24³ subtomogram convention, wedge-safe augmentation); Module 04 (k-means, ARI, Hungarian accuracy); Module 05 (PyTorch) |
| **Estimated time** | 150 min (reading + running + ✏️ exercises) |
| **Lab papers covered** | Zeng *et al.* 2018 (autoencoder features); Liu S *et al.* 2019 (semi-supervised autoencoding classifier, from code); Gupta *et al.* 2022 (self-supervised learning); Li R *et al.* 2020 (ProtoNet3D / ProtoNet-CE); Gao *et al.* 2022 (FSCC); Du *et al.* 2021 (HAL active learning); Du *et al.* 2019 (open-set, Soft LMCCL); cited only: Wang T *et al.* 2022, Zeng Y *et al.* 2021, AL-GradNorm 2022, noise-stability AL 2024 — see [`PAPER_ATLAS.md`](../PAPER_ATLAS.md) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> All experiments here are small, controlled *toy versions* of published methods, designed to teach the core idea of
> each paper on a laptop CPU in minutes. They do **not** reproduce the papers' architectures, datasets or numbers.
> Molecular densities come from real PDB entries (via the open-source PolNet simulator); if you are offline the
> notebook falls back to simple stand-in shapes and says so.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Train** a 3D convolutional autoencoder without labels, **cluster** its codes with k-means and **explain** why
   orientation, not identity, often dominates unsupervised features.
2. **Implement** a SimCLR-style contrastive loss with cryo-ET-aware augmentations and **measure** with a linear probe how
   much 1, 3 or 10 labels per class are worth after self-supervised pre-training.
3. **Train** a prototypical network episodically and **evaluate** it on molecules it has never seen (5-way, 1- and
   5-shot); **implement** FSCC-style distribution calibration.
4. **Run** an active-learning loop and **compare** random, uncertainty (entropy) and hybrid HAL-style querying with
   learning curves.
5. **Build** an open-set embedding with a cosine-margin loss plus center loss and **detect** molecules that were not in
   the training set (ClpB and p97).
6. **Choose**, for a given biological situation, which few-label strategy fits — and state its assumptions.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | toolkit, templates, one cached dataset (known + novel molecules) | 10 min |
| 1 · Features without labels | autoencoder → k-means; a semi-supervised autoencoding classifier | 25 min |
| 2 · Contrastive self-supervision | SimCLR-style pre-training → linear probe with 1–10 labels | 25 min |
| 3 · Few-shot learning | prototypical networks; FSCC distribution calibration | 30 min |
| 4 · Active learning | which particles should the expert label next? | 20 min |
| 5 · Open-set recognition | train on six molecules, recognise "none of the above" | 25 min |
| ✅ / 🧠 / 🚀 | checkpoint, discussion, going further | 15 min |

## 🧫 The biological question

A structural biologist opens a new tomogram of a neuron. Thousands of particles are visible. She can recognise a few
dozen with confidence — the ribosomes, perhaps a proteasome — and labelling each one takes time and expertise. Worse,
the cell almost certainly contains complexes that are **in no training set**: a chaperone caught in the act, a
disease-related aggregate, a molecule no one has catalogued. Supervised classifiers (Module 15) need hundreds of
labelled examples per class and assume every particle belongs to a known class. This notebook covers the lab's answers
to four practical questions:

| question | technique | section |
|---|---|---|
| Can unlabelled particles teach a network anything? | autoencoders, contrastive self-supervision | 1–2 |
| Can I recognise a new molecule from 1–5 examples? | few-shot learning (prototypes, calibration) | 3 |
| Which particles should the expert label first? | active learning | 4 |
| Can the network say "I have never seen this"? | open-set recognition | 5 |

---
## 0 · Environment setup

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, this notebook is designed for a CPU.")
try:
    import torch
    print("PyTorch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
except ImportError:
    print("PyTorch not installed yet — the next cell installs it.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("matplotlib", "matplotlib"),
                         ("sklearn", "scikit-learn"), ("torch", "torch")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import math
import os
import time
import warnings
from itertools import combinations

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy import ndimage
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import adjusted_rand_score, roc_auc_score, roc_curve

warnings.filterwarnings("ignore", category=UserWarning)
SEED = 0
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "image.cmap": "gray"})
print("NumPy", np.__version__, "| PyTorch", torch.__version__, "| device:", DEVICE, "| seed", SEED)

### 🧰 Course toolkit
Copied verbatim from [`course_toolkit.py`](../course_toolkit.py) so that this notebook runs on its own.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v1.0; copied so this notebook runs alone) ──
import struct
import urllib.request

POLNET_COMMIT = "73e613cc9162d26c7fd740ce3299bcb280a3e3b6"
POLNET_URL = ("https://raw.githubusercontent.com/anmartinezs/polnet/"
              + POLNET_COMMIT + "/data/templates/{kind}/{name}.{ext}")
CACHE_DIR = os.environ.get("CRYO_COURSE_CACHE", "course_data")

ZOO = {
    "4v4r": dict(name="70S ribosome", organism="bacterial (E. coli / T. thermophilus)", mass_kDa=2277, symmetry="C1"),
    "4v7r": dict(name="80S ribosome", organism="S. cerevisiae", mass_kDa=5851, symmetry="C1"),
    "5mrc": dict(name="mitochondrial ribosome", organism="S. cerevisiae", mass_kDa=3325, symmetry="C1"),
    "3j9i": dict(name="20S proteasome", organism="Thermoplasma acidophilum", mass_kDa=659, symmetry="D7"),
    "6utj": dict(name="singly capped proteasome", organism="Thermoplasma acidophilum", mass_kDa=839, symmetry="C7"),
    "4cr2": dict(name="26S proteasome", organism="S. cerevisiae", mass_kDa=1309, symmetry="C1"),
    # NB: PolNet's 4V94 map holds TWO TRiC complexes side by side (1.95 MDa ≈ 2 × ~1 MDa).
    "4v94": dict(name="TRiC/CCT chaperonin (×2)", organism="S. cerevisiae", mass_kDa=1952, symmetry="C2"),
    "1bxn": dict(name="RuBisCO", organism="Cupriavidus necator", mass_kDa=280, symmetry="D4"),
    "1qvr": dict(name="ClpB chaperone", organism="Thermus thermophilus", mass_kDa=297, symmetry="C1"),
    "3cf3": dict(name="p97/VCP ATPase", organism="Mus musculus", mass_kDa=None, symmetry="C6"),
    "2cg9": dict(name="Hsp90–Sba1 chaperone", organism="S. cerevisiae", mass_kDa=None, symmetry="C2"),
    "1u6g": dict(name="Cand1–Cul1–Roc1 ligase", organism="H. sapiens", mass_kDa=239, symmetry="C1"),
    "3d2f": dict(name="Sse1p–Hsp70 chaperone", organism="S. cerevisiae / H. sapiens", mass_kDa=236, symmetry="C1"),
    "1s3x": dict(name="Hsp70 ATPase domain", organism="H. sapiens", mass_kDa=43, symmetry="C1"),
}


def _download(url: str, dest: str, timeout: float = 60.0) -> bool:
    """Download ``url`` to ``dest`` (no-op if it exists). Returns True on success."""
    if os.path.exists(dest) and os.path.getsize(dest) > 0:
        return True
    os.makedirs(os.path.dirname(dest) or ".", exist_ok=True)
    try:
        with urllib.request.urlopen(url, timeout=timeout) as r, open(dest + ".part", "wb") as f:
            f.write(r.read())
        os.replace(dest + ".part", dest)
        return True
    except Exception as exc:  # network blocked, offline, ...
        print(f"  ⚠️  download failed for {url.split('/')[-1]}: {type(exc).__name__}")
        return False


def read_mrc(path_or_bytes) -> tuple[np.ndarray, float]:
    """Minimal MRC2014 reader (modes 0, 1, 2, 6). Returns (array[z, y, x], voxel_size_Å).

    The MRC header is 1024 bytes of little-endian 32-bit words; words 1–3 are the
    grid size (nx, ny, nz), word 4 the data type, words 11–13 the cell size in Å,
    and word 24 the length of an optional extended header.
    """
    raw = path_or_bytes if isinstance(path_or_bytes, (bytes, bytearray)) else open(path_or_bytes, "rb").read()
    if raw[:2] == b"\x1f\x8b":  # gzip
        import gzip
        raw = gzip.decompress(raw)
    nx, ny, nz, mode = struct.unpack("<4i", raw[0:16])
    mx, my, mz = struct.unpack("<3i", raw[28:40])
    cella = struct.unpack("<3f", raw[40:52])
    mapc, mapr, maps = struct.unpack("<3i", raw[64:76])
    nsymbt = struct.unpack("<i", raw[92:96])[0]
    dtype = {0: np.int8, 1: np.int16, 2: np.float32, 6: np.uint16}[mode]
    data = np.frombuffer(raw, dtype=dtype, offset=1024 + nsymbt, count=nx * ny * nz)
    vol = data.reshape(nz, ny, nx).astype(np.float32)
    if (mapc, mapr, maps) != (1, 2, 3) and (mapc, mapr, maps) != (0, 0, 0):
        # reorder axes so that the array is [z, y, x]
        order = [3 - maps, 3 - mapr, 3 - mapc]
        vol = np.transpose(vol, np.argsort(order))
    voxel = float(cella[0] / mx) if mx > 0 else 1.0
    return vol, voxel


def procedural_molecule(pdb_id: str, size: int = 40) -> np.ndarray:
    """Offline fallback: a coarse shape with the right size/symmetry flavour (1 nm voxels)."""
    g = (np.arange(size) - size / 2 + 0.5)
    z, y, x = np.meshgrid(g, g, g, indexing="ij")
    r = np.sqrt(x**2 + y**2)

    def blob(cz, cy, cx, s):
        return np.exp(-((z - cz)**2 + (y - cy)**2 + (x - cx)**2) / (2 * s**2))
    pid = pdb_id.lower()
    if pid in ("4v4r", "4v7r", "5mrc"):          # two unequal subunits
        v = blob(-3, 0, 0, 7.5) + 0.8 * blob(6, 2, 1, 5.5)
    elif pid in ("3j9i", "6utj", "4cr2"):        # stacked rings / barrel
        v = np.exp(-((r - 5.5)**2) / 4.0) * (np.abs(z) < 7.5)
        if pid != "3j9i":
            v = v + 0.8 * blob(10, 0, 0, 4)
    elif pid == "4v94":                          # double ring with cavity
        v = np.exp(-((r - 7)**2) / 6.0) * (np.abs(z) < 8)
    elif pid in ("3cf3", "1qvr"):                # hexameric ring
        ang = np.arctan2(y, x)
        v = np.exp(-((r - 5)**2) / 3.0) * (np.abs(z) < 4) * (1 + 0.5 * np.cos(6 * ang))
    elif pid == "1bxn":                          # compact ball
        v = blob(0, 0, 0, 4.5)
    else:                                        # small asymmetric blob
        v = blob(0, 0, 0, 3) + 0.5 * blob(2, 2, 0, 2)
    return ndimage.gaussian_filter(v.astype(np.float32), 1.0)


def fetch_zoo_map(pdb_id: str, voxel: str = "10A") -> tuple[np.ndarray, float, str]:
    """Return (density[z, y, x], voxel_size_Å, source) for a molecular-zoo entry.

    Tries the pinned PolNet template first; if the download fails (no internet),
    falls back to a *procedural* stand-in so the notebook still runs.
    """
    pdb_id = pdb_id.lower()
    dest = os.path.join(CACHE_DIR, f"polnet_{voxel}_{pdb_id}.mrc")
    url = POLNET_URL.format(kind=f"mrcs_{voxel}", name=pdb_id, ext="mrc")
    if _download(url, dest):
        vol, vs = read_mrc(dest)
        return vol, vs, f"PolNet template from PDB {pdb_id.upper()}"
    vol = procedural_molecule(pdb_id)
    return vol, 10.0, f"procedural stand-in for {pdb_id.upper()} (offline fallback)"


def prepare_map(vol: np.ndarray, size: int = 32, sigma: float = 1.0) -> np.ndarray:
    """Low-pass, centre (by centre of mass), crop/pad to ``size``³ and scale to [0, 1].

    PolNet templates are sharp sums of atoms; a Gaussian blur of ~1 voxel (1 nm)
    mimics the resolution of a cellular tomogram.
    """
    v = ndimage.gaussian_filter(np.clip(vol, 0, None).astype(np.float32), sigma)
    com = np.array(ndimage.center_of_mass(v))
    shift = np.array(v.shape) / 2 - 0.5 - com
    v = ndimage.shift(v, shift, order=1)
    out = np.zeros((size, size, size), np.float32)
    s = np.array(v.shape)
    src0 = np.maximum((s - size) // 2, 0)
    dst0 = np.maximum((size - s) // 2, 0)
    n = np.minimum(s, size)
    out[dst0[0]:dst0[0] + n[0], dst0[1]:dst0[1] + n[1], dst0[2]:dst0[2] + n[2]] = \
        v[src0[0]:src0[0] + n[0], src0[1]:src0[1] + n[1], src0[2]:src0[2] + n[2]]
    out -= out.min()
    return out / (out.max() + 1e-12)


def random_rotation(rng: np.random.Generator) -> np.ndarray:
    """Uniformly random proper rotation (det = +1) from a random unit quaternion."""
    q = rng.normal(size=4)
    q /= np.linalg.norm(q)
    w, x, y, z = q
    return np.array([
        [1 - 2 * (y * y + z * z), 2 * (x * y - z * w), 2 * (x * z + y * w)],
        [2 * (x * y + z * w), 1 - 2 * (x * x + z * z), 2 * (y * z - x * w)],
        [2 * (x * z - y * w), 2 * (y * z + x * w), 1 - 2 * (x * x + y * y)]])


_P = np.array([[0, 0, 1], [0, 1, 0], [1, 0, 0]], float)  # (x,y,z) <-> (z,y,x)


def rotate_volume(vol: np.ndarray, R: np.ndarray, shift_xyz=(0, 0, 0), order: int = 1,
                  cval: float = 0.0) -> np.ndarray:
    """Rotate a volume about its centre by R (then translate by shift, in voxels, (x,y,z)).

    out(p) = vol(Rᵀ (p − c − t) + c): the *object* is rotated by R and moved by t.
    """
    R_idx = _P @ R @ _P                       # same rotation written in (z, y, x) index order
    c = (np.array(vol.shape) - 1) / 2.0
    t_idx = _P @ np.asarray(shift_xyz, float)
    M = R_idx.T                               # inverse rotation maps output -> input
    offset = c - M @ (c + t_idx)
    return ndimage.affine_transform(vol, M, offset=offset, order=order, mode="constant", cval=cval)


def freq_grid(shape):
    """Frequency coordinates (cycles/voxel) for an unshifted FFT; returns list per axis."""
    return np.meshgrid(*[np.fft.fftfreq(n) for n in shape], indexing="ij")


def missing_wedge_mask(shape, tilt_max_deg: float = 60.0) -> np.ndarray:
    """Binary Fourier mask of a single-axis tilt series about y (±tilt_max).

    A coefficient (k_z, k_y, k_x) is measured if |k_z| ≤ |k_x|·tan(tilt_max)
    — the formula M(ξ) = 1(|ξ_3| ≤ |ξ_1| tan θ) used by Xu et al. (2012).
    For 2D images [z, x] pass a 2-tuple shape.
    """
    k = freq_grid(shape)
    kz, kx = k[0], k[-1]
    m = np.abs(kz) <= np.abs(kx) * np.tan(np.radians(tilt_max_deg)) + 1e-9
    m[(0,) * len(shape)] = True               # keep the mean (DC) term
    return m.astype(np.float32)


def apply_fourier_mask(vol: np.ndarray, mask: np.ndarray) -> np.ndarray:
    """Multiply the FFT of ``vol`` by ``mask`` (unshifted layout) and return the real part."""
    return np.real(np.fft.ifftn(np.fft.fftn(vol) * mask)).astype(np.float32)


def add_noise_snr(signal: np.ndarray, snr: float, rng: np.random.Generator) -> np.ndarray:
    """Add white Gaussian noise so that var(signal)/var(noise) = snr."""
    sigma = np.sqrt(signal.var() / snr)
    return (signal + rng.normal(0, sigma, signal.shape)).astype(np.float32)


def normalize(v: np.ndarray) -> np.ndarray:
    """Zero mean, unit standard deviation."""
    return ((v - v.mean()) / (v.std() + 1e-12)).astype(np.float32)


def simulate_subtomogram(template: np.ndarray, rng: np.random.Generator, snr: float = 0.1,
                         tilt_max_deg: float | None = 60.0, max_shift: float = 0.0,
                         R: np.ndarray | None = None):
    """Randomly rotate/shift a template, impose a missing wedge and add noise.

    Returns (subtomogram, R, shift_xyz).  This 'direct' simulation (Fourier mask +
    white noise) is the fast approximation used by many Xu-lab benchmarks.
    """
    R = random_rotation(rng) if R is None else R
    t = rng.uniform(-max_shift, max_shift, 3) if max_shift > 0 else np.zeros(3)
    v = rotate_volume(template, R, t)
    if tilt_max_deg is not None:
        v = apply_fourier_mask(v, missing_wedge_mask(v.shape, tilt_max_deg))
    return add_noise_snr(v, snr, rng), R, t


def hungarian_accuracy(y_true, y_pred) -> float:
    """Clustering accuracy after the best one-to-one relabelling (Hungarian algorithm)."""
    from scipy.optimize import linear_sum_assignment
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    ct = np.zeros((y_pred.max() + 1, y_true.max() + 1))
    np.add.at(ct, (y_pred, y_true), 1)
    r, c = linear_sum_assignment(-ct)
    return float(ct[r, c].sum() / len(y_true))


print("🧰 toolkit loaded: fetch_zoo_map, prepare_map, simulate_subtomogram, hungarian_accuracy, …")

### 0.1 One cached dataset: six known and five novel molecules

Same conventions as Module 15: zoo templates binned to **15 Å voxels** in **24³ boxes** (36 nm), random orientation,
±1-voxel shifts, a ±60° missing wedge and white noise. We use a single noise level, **SNR 0.1**, throughout (✏️ try
0.03 at the end).

| set | molecules | per class | role |
|---|---|---|---|
| `POOL` | 6 **known**: 70S & 80S ribosomes, 20S, capped and 26S proteasomes, RuBisCO | 120 | unlabelled pool / base-class training data |
| `TEST` | the same 6 known | 50 | evaluation |
| `NOVEL` | 5 **novel**: ClpB, p97, mitoribosome, TRiC (×2), Hsp90 | 60 | few-shot and open-set evaluation — never used for training |

In [ ]:
# ── 0.1 Templates and the cached dataset ──
VOXEL_A, BOX = 15.0, 24
KNOWN = ["4v4r", "4v7r", "3j9i", "6utj", "4cr2", "1bxn"]
NOVEL = ["1qvr", "3cf3", "5mrc", "4v94", "2cg9"]
SHORT = {"4v4r": "70S ribo", "4v7r": "80S ribo", "3j9i": "20S prot", "6utj": "capped prot", "4cr2": "26S prot",
         "1bxn": "RuBisCO", "1qvr": "ClpB", "3cf3": "p97", "5mrc": "mitoribo", "4v94": "TRiC×2", "2cg9": "Hsp90"}
K_NAMES, N_NAMES = [SHORT[p] for p in KNOWN], [SHORT[p] for p in NOVEL]


def make_template(pid: str, box: int = BOX, voxel_A: float = VOXEL_A) -> np.ndarray:
    """Zoo map → box³ template at voxel_A Å/voxel (prepared at 10 Å, then resampled), scaled to [0, 1]."""
    raw, vs, src = fetch_zoo_map(pid)
    n10 = int(round(box * voxel_A / 10.0))
    v = ndimage.zoom(prepare_map(raw, size=n10, sigma=1.0), box / n10, order=1)
    v = np.clip(v, 0, None)
    print(f"  {pid.upper()} {ZOO[pid]['name']:26s} ← {src}")
    return (v / v.max()).astype(np.float32)


def simulate_set(pids, n_per_class, rng, snr=0.1):
    """n_per_class subtomograms per molecule (normalised) → (X [N, 1, 24, 24, 24], y [N])."""
    X, y = [], []
    for k, pid in enumerate(pids):
        for _ in range(n_per_class):
            v, _, _ = simulate_subtomogram(TEMPLATES[pid], rng, snr=snr, tilt_max_deg=60, max_shift=1.0)
            X.append(normalize(v)); y.append(k)
    return np.stack(X)[:, None], np.array(y)


TEMPLATES = {pid: make_template(pid) for pid in KNOWN + NOVEL}
t0 = time.time()
data_rng = np.random.default_rng(SEED)
X_pool, y_pool = simulate_set(KNOWN, 120, data_rng)
X_test, y_test = simulate_set(KNOWN, 50, data_rng)
X_nov, y_nov = simulate_set(NOVEL, 60, data_rng)
print(f"pool {X_pool.shape}, test {X_test.shape}, novel {X_nov.shape} — simulated in {time.time() - t0:.0f} s")

fig, axes = plt.subplots(2, 11, figsize=(11, 2.3))
for j, pid in enumerate(KNOWN + NOVEL):
    t = TEMPLATES[pid]
    axes[0, j].imshow(max([t.sum(a) for a in range(3)], key=lambda p: (p > 0.2 * p.max()).sum()))
    Xs, ys = (X_pool, y_pool) if pid in KNOWN else (X_nov, y_nov)
    k = (KNOWN if pid in KNOWN else NOVEL).index(pid)
    axes[1, j].imshow(Xs[np.where(ys == k)[0][0], 0, BOX // 2])
    axes[0, j].set_title(("known: " if pid in KNOWN else "novel: ") + SHORT[pid], fontsize=6.5)
for ax in axes.ravel():
    ax.axis("off")
fig.suptitle("Top: clean templates (side projections). Bottom: one noisy subtomogram slice (SNR 0.1)", fontsize=9)
plt.tight_layout(); plt.show()

### 0.2 Shared building blocks

All networks in this notebook use the same small **3D encoder** (stride-2 stem → 12³ → pool → 6³ → pool → 3³), which
ends with **global average pooling**: the 64 channels are averaged over the 3³ positions, giving one 64-number
*feature vector* per subtomogram. Averaging over positions makes the features less sensitive to exactly *where* in the
box a structure appears — useful when labels are scarce.

In [ ]:
# ── 0.2 Encoder, wedge-safe augmentation, and feature extraction ──
def conv_bn_relu(c_in, c_out, stride=1):
    """3×3×3 convolution → batch normalisation → ReLU."""
    return nn.Sequential(nn.Conv3d(c_in, c_out, 3, stride=stride, padding=1), nn.BatchNorm3d(c_out), nn.ReLU())


class Encoder3D(nn.Module):
    """24³ subtomogram → 64-number feature vector (conv stack + global average pooling)."""

    def __init__(self, w=8, out_dim=64):
        super().__init__()
        self.conv = nn.Sequential(conv_bn_relu(1, w, stride=2), conv_bn_relu(w, 2 * w), nn.MaxPool3d(2),
                                  conv_bn_relu(2 * w, 4 * w), nn.MaxPool3d(2), conv_bn_relu(4 * w, out_dim))
        self.out_dim = out_dim

    def forward(self, x):
        return self.conv(x).mean(dim=(2, 3, 4))


def wedge_safe_flip(x, gen):
    """Random 180° rotation about z, y or x (or none) for each volume of a batch [B, 1, z, y, x] (Module 15)."""
    out = x.clone()
    choice = torch.randint(0, 4, (len(x),), generator=gen)
    for c, dims in {1: (3, 4), 2: (2, 4), 3: (2, 3)}.items():
        sel = choice == c
        if sel.any():
            out[sel] = torch.flip(x[sel], dims)
    return out


@torch.no_grad()
def embed(model, X, batch=128) -> np.ndarray:
    """Features (or any vector output) of `model` for every volume, in eval mode."""
    model.eval()
    return torch.cat([model(torch.as_tensor(X[i:i + batch]).to(DEVICE)).cpu()
                      for i in range(0, len(X), batch)]).numpy()


def n_params(m) -> int:
    """Number of trainable weights."""
    return sum(p.numel() for p in m.parameters() if p.requires_grad)


print(f"Encoder3D: {n_params(Encoder3D()):,} weights → features of size {Encoder3D()(torch.zeros(2, 1, BOX, BOX, BOX)).shape}")

---
## 1 · Features without labels: autoencoders (Zeng *et al.* 2018)

### 📍 Where we are
Module 15 trained classifiers with hundreds of labels per class; Module 16 squeezed segmentation out of image-level
labels. Now we go further down the supervision ladder — to **no labels at all**, then to just a few.

An **autoencoder** is a network that must reproduce its input after squeezing it through a narrow "bottleneck": an
**encoder** $f$ maps the 13,824-voxel box $x$ to a short **code** $z = f(x)$ (here 32 numbers), and a **decoder** $g$
maps the code back to a box $\hat x = g(z)$. Training minimises the reconstruction error

$$\mathcal L_{AE} = \frac{1}{N}\sum_{i} \lVert x_i - g(f(x_i)) \rVert^2 \;+\; \beta \sum_i \lVert z_i \rVert_1 .$$

*In words:* to rebuild the box from only 32 numbers, the code must capture its most important content — size, shape,
position, orientation — while the small L1 penalty (β) keeps codes sparse, as in Zeng *et al.* No labels are used.
The paper then clusters the codes with k-means and lets a human inspect the *decoded cluster centres*.

In [ ]:
# ── 1.1 A 3D convolutional autoencoder ──
class AutoEncoder3D(nn.Module):
    """Encoder (conv stack → 32-number code) and a mirrored decoder (transposed convolutions back to 24³)."""

    def __init__(self, w=8, code=32):
        super().__init__()
        self.conv = nn.Sequential(conv_bn_relu(1, w, stride=2), conv_bn_relu(w, 2 * w), nn.MaxPool3d(2),
                                  conv_bn_relu(2 * w, 4 * w), nn.MaxPool3d(2))        # → [4w, 3, 3, 3]
        self.to_code = nn.Linear(4 * w * 27, code)
        self.from_code = nn.Linear(code, 4 * w * 27)
        self.deconv = nn.Sequential(
            nn.ConvTranspose3d(4 * w, 2 * w, 2, stride=2), nn.BatchNorm3d(2 * w), nn.ReLU(),     # 3³ → 6³
            nn.ConvTranspose3d(2 * w, w, 2, stride=2), nn.BatchNorm3d(w), nn.ReLU(),             # 6³ → 12³
            nn.ConvTranspose3d(w, 1, 2, stride=2))                                              # 12³ → 24³
        self.w = w

    def encode(self, x):
        return F.relu(self.to_code(self.conv(x).flatten(1)))

    def decode(self, z):
        return self.deconv(F.relu(self.from_code(z)).view(-1, 4 * self.w, 3, 3, 3))

    def forward(self, x):
        z = self.encode(x)
        return self.decode(z), z


def train_autoencoder(model, X, epochs=6, batch=32, lr=2e-3, beta=1e-4, X_lab=None, y_lab=None, seed=SEED):
    """Minimise MSE(x, x̂) + β·|z|₁ on all of X; if labels are given, add cross-entropy on 8 labelled boxes per batch."""
    torch.manual_seed(seed); gen = torch.Generator().manual_seed(seed)
    model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    steps = epochs * math.ceil(len(X) / batch)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=steps)
    Xt = torch.as_tensor(X); t0 = time.time()
    for ep in range(epochs):
        model.train(); perm = torch.randperm(len(X), generator=gen); total = 0.0
        for i in range(0, len(X), batch):
            xb = wedge_safe_flip(Xt[perm[i:i + batch]], gen).to(DEVICE)
            n_lab = 0
            if X_lab is not None:                          # semi-supervised: append labelled boxes
                j = torch.randint(0, len(X_lab), (8,), generator=gen)
                xb = torch.cat([xb, wedge_safe_flip(torch.as_tensor(X_lab[j]), gen).to(DEVICE)])
                n_lab = 8
            recon, z = model(xb)
            loss = F.mse_loss(recon, xb) + beta * z.abs().sum(1).mean()
            if n_lab:
                loss = loss + F.cross_entropy(model.classify(xb[-n_lab:]), torch.as_tensor(y_lab[j]).to(DEVICE))
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
            total += loss.item()
        print(f"  epoch {ep + 1}/{epochs}  loss {total / math.ceil(len(X) / batch):.3f}  ({time.time() - t0:.0f} s)")
    return model


torch.manual_seed(SEED)
ae = train_autoencoder(AutoEncoder3D(), X_pool, epochs=6)

In [ ]:
# ── 1.2 Cluster the codes with k-means; compare with raw voxels ──
class _Enc(nn.Module):
    """Wrap any callable feature function as a module, so `embed` can switch it to eval mode."""

    def __init__(self, owner, fn):
        super().__init__()
        self.owner, self.fn = owner, fn

    def forward(self, x):
        return self.fn(x)


codes_pool = embed(_Enc(ae, ae.encode), X_pool)
raw_pca = PCA(n_components=32, random_state=SEED).fit_transform(X_pool.reshape(len(X_pool), -1))
cluster_results = {}
for name, Z in [("autoencoder codes (32-d)", codes_pool), ("raw voxels (PCA-32)", raw_pca)]:
    labels = KMeans(n_clusters=len(KNOWN), n_init=10, random_state=SEED).fit_predict(Z)
    cluster_results[name] = labels
    print(f"k-means (k = 6) on {name:24s}: Hungarian accuracy {hungarian_accuracy(y_pool, labels):.2f}  "
          f"ARI {adjusted_rand_score(y_pool, labels):.2f}")

# what does a cluster centre look like? decode it
km12 = KMeans(n_clusters=12, n_init=10, random_state=SEED).fit(codes_pool)
with torch.no_grad():
    ae.eval()
    centres = ae.decode(torch.as_tensor(km12.cluster_centers_, dtype=torch.float32).to(DEVICE)).cpu().numpy()[:, 0]
fig, axes = plt.subplots(2, 6, figsize=(8, 3))
for j, ax in enumerate(axes.ravel()):
    members = y_pool[km12.labels_ == j]
    top = np.bincount(members, minlength=len(KNOWN))
    ax.imshow(centres[j].sum(0)); ax.axis("off")
    ax.set_title(f"n={len(members)}: {K_NAMES[top.argmax()]}\n({100 * top.max() / max(len(members), 1):.0f} % pure)", fontsize=7)
fig.suptitle("Decoded k-means centres (k = 12) of autoencoder codes — what a user would inspect", fontsize=9)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Without a single label, the autoencoder's codes group the subtomograms better than raw
voxels do — the decoded centres look like blurry molecules of different sizes. But k-means on the codes is far from
perfect: to *reconstruct* a box, the code must also store **where** and **in which orientation** the molecule is, and
k-means happily groups "large blob tilted left" regardless of identity. Zeng *et al.* therefore offered an optional
**pose normalisation** (shift to the centre of mass, rotate to the principal axes) and **over-clustered** (k = 100),
leaving it to a human to tag the decoded centres. Separating *what* from *how it is posed* is a theme of Modules 19–21.

### 1.3 A semi-supervised autoencoding classifier (Liu S *et al.* 2019)

If we have **some** labels, we can add a classification head to the same encoder and train both objectives at once:
reconstruction on *all* boxes, cross-entropy only on the *labelled* ones,

$$\mathcal L = \frac{1}{|U \cup L|}\sum_{x \in U\cup L} \lVert x - \hat x \rVert^2 + \frac{1}{|L|}\sum_{(x, y) \in L} \mathrm{CE}\big(y, p(x)\big).$$

*In words:* the unlabelled boxes keep the encoder honest (it must describe them well), while the few labelled boxes
point it towards the distinctions that matter. We compare it with the same network trained on the labelled boxes
alone, using **10 labels per class** (60 of the 720 pool subtomograms).

In [ ]:
# ── 1.3 Supervised-only vs semi-supervised with 10 labels per class ──
class AutoEncodingClassifier(AutoEncoder3D):
    """AutoEncoder3D plus a classification head on the flattened encoder features."""

    def __init__(self, n_classes=6, w=8):
        super().__init__(w=w)
        self.cls_head = nn.Sequential(nn.BatchNorm1d(4 * w * 27), nn.Linear(4 * w * 27, 64), nn.ReLU(),
                                      nn.Linear(64, n_classes))

    def classify(self, x):
        return self.cls_head(self.conv(x).flatten(1))


lab_rng = np.random.default_rng(SEED)
lab_idx = np.concatenate([lab_rng.choice(np.where(y_pool == k)[0], 10, replace=False) for k in range(len(KNOWN))])
X_lab, y_lab = X_pool[lab_idx], y_pool[lab_idx]


def train_supervised_only(model, X, y, steps=270, batch=16, lr=2e-3, seed=SEED):
    """Cross-entropy on the labelled boxes only (with wedge-safe flips)."""
    torch.manual_seed(seed); gen = torch.Generator().manual_seed(seed)
    model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=steps)
    for _ in range(steps):
        model.train()
        j = torch.randint(0, len(X), (batch,), generator=gen)
        loss = F.cross_entropy(model.classify(wedge_safe_flip(torch.as_tensor(X[j]), gen).to(DEVICE)),
                               torch.as_tensor(y[j]).to(DEVICE))
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    return model


@torch.no_grad()
def classify_acc(model, X, y) -> float:
    """Accuracy of model.classify on (X, y)."""
    model.eval()
    p = torch.cat([model.classify(torch.as_tensor(X[i:i + 128]).to(DEVICE)).cpu() for i in range(0, len(X), 128)])
    return float((p.argmax(1).numpy() == y).mean())


t0 = time.time()
torch.manual_seed(SEED)
sup_only = train_supervised_only(AutoEncodingClassifier(), X_lab, y_lab)
print(f"supervised only (60 labels)           : test accuracy {classify_acc(sup_only, X_test, y_test):.2f}  ({time.time() - t0:.0f} s)")
t0 = time.time()
torch.manual_seed(SEED)
semi = train_autoencoder(AutoEncodingClassifier(), X_pool, epochs=4, X_lab=X_lab, y_lab=y_lab)
print(f"semi-supervised (60 labels + 720 unlabelled): test accuracy {classify_acc(semi, X_test, y_test):.2f}  "
      f"({time.time() - t0:.0f} s)")

🔍 **What just happened?** Both networks saw the same 60 labels; the semi-supervised one also had to reconstruct 720
unlabelled boxes. Whether that helps depends on how much the reconstruction task and the classification task have in
common — here both need size and shape, so the extra data usually helps a little; with stronger noise or very
different molecules it may not. Liu S *et al.*'s released code uses exactly this recipe (1:1 weights, a per-sample mask
that applies the cross-entropy only to labelled boxes).

### 📄 Paper connection — autoencoder features (Zeng *et al.* 2018) and the autoencoding classifier (Liu S *et al.* 2019)

**Paper 1**: Zeng X, Leung MR, Zeev-Ben-Mordehai T, Xu M. *A convolutional autoencoder approach for mining features in
cellular electron cryo-tomograms and weakly supervised coarse segmentation.* J Struct Biol 202(2):150–160 (2018).
[doi:10.1016/j.jsb.2017.12.015](https://doi.org/10.1016/j.jsb.2017.12.015) · [arXiv:1706.04970](https://arxiv.org/abs/1706.04970)

- **Question it answers:** which kinds of structures (membranes, large globular complexes, …) exist in a cellular
  tomogram, found without labels?
- **What the paper actually does:** DoG-picked 32³ subvolumes → **Encoder3D** (two 3³ convolutions, each followed by
  2³ max-pooling, then a fully connected 32-d code; ReLU; L1 sparsity on the code) and a mirrored **Decoder3D**, trained
  with mean-squared reconstruction error; optional pose normalisation (centre of mass + principal axes); **k-means
  (k = 100)** on the codes; each centre is decoded and tagged by a user (surface / large globular / not selected) — the
  tags then train the weakly supervised segmenter EDSS3D (Module 16).
- **What our toy keeps:** a 3D conv autoencoder with a 32-d sparse code, k-means, decoded centres.
- **What our toy leaves out:** DoG picking from a real tomogram, pose normalisation, k = 100, the segmentation stage.
- **Result reported:** on a simulated tomogram, the selected ribosome clusters gave precision 66.0 % and recall
  61.3 %; surface clusters 73.9 % and 44.7 % (real COS-7 tomograms were assessed visually).
- **Code:** AITom `aitom/classify/deep/unsupervised/autoencoder/` (`autoencoder.py`, `seg_src.py`).

**Paper 2**: Liu S, Du X, Xi R, Xu F, Zeng X, Zhou B, Xu M. *Semi-supervised macromolecule structural classification
in cellular electron cryo-tomograms using 3D autoencoding classifier.* BMVC 2019.

> ⚠️ We could not access the full text of this paper while writing; the description below comes from the lab's released
> code (AITom `aitom/classify/deep/supervised/cnn/autoencoding/`). Read the paper before relying on details; its data
> and results are not given here.

- **What the code does:** one shared 3D conv encoder (Conv 32@3³ → max-pool → Conv 32@3³ → max-pool) with two heads — a
  decoder (dense layers → reshape → 2 × [upsampling + conv] → linear 1-channel conv) and a classifier (batch norm →
  Dense 128 → Dense 128 → softmax); joint loss = MSE reconstruction + categorical cross-entropy (weights 1:1), with
  per-sample masks so that the cross-entropy uses labelled samples only (the equation above is reconstructed from the
  code).

---
## 2 · Contrastive self-supervision (SimCLR-style; Gupta *et al.* 2022)

An autoencoder must remember *everything* about a box, including its pose. **Contrastive learning** asks a different
question: *are these two boxes two views of the same particle?* We create two random **views** $\tilde x_i, \tilde x_j$
of each subtomogram by augmentation, embed them ($z = g(f(x))$, with an encoder $f$ and a small *projection head* $g$),
and train with the **NT-Xent / InfoNCE** loss (SimCLR; Chen *et al.* 2020):

$$\ell_{i,j} = -\log \frac{\exp(\mathrm{sim}(z_i, z_j)/\tau)}{\sum_{k \neq i} \exp(\mathrm{sim}(z_i, z_k)/\tau)},
\qquad \mathrm{sim}(a, b) = \frac{a \cdot b}{\lVert a\rVert\,\lVert b\rVert}.$$

*In words:* in a batch of $2B$ views, each view must pick out its partner among $2B - 1$ candidates; the temperature
$\tau$ sets how sharply. Whatever the augmentations change (rotation, shift, noise), the features learn to **ignore** —
so the choice of augmentations *is* the choice of what counts as "the same molecule".

**Cryo-ET-aware augmentations** (following Gupta *et al.*: rotation about z by up to ±45°, translation, scaling; plus
our wedge-safe 180° flips, extra noise and contrast jitter). Rotations about the beam axis z keep the missing wedge
around z, so the views remain plausible tomogram data.

In [ ]:
# ── 2.1 Random views with PyTorch's differentiable resampler ──
def random_views(x, gen, max_deg=45.0, max_shift_vox=2.0):
    """Augment a batch [B, 1, z, y, x]: rotation about z (±max_deg), shift, scale 0.9–1.1, 180° flips,
    added noise and contrast jitter; each volume is re-normalised."""
    B = len(x)
    a = (torch.rand(B, generator=gen) * 2 - 1) * math.radians(max_deg)
    s = 0.9 + 0.2 * torch.rand(B, generator=gen)
    t = (torch.rand(B, 3, generator=gen) * 2 - 1) * max_shift_vox / (BOX / 2)   # in [-1, 1] grid units
    theta = torch.zeros(B, 3, 4)
    theta[:, 0, 0], theta[:, 0, 1] = torch.cos(a) / s, -torch.sin(a) / s          # grid coordinates are (x, y, z)
    theta[:, 1, 0], theta[:, 1, 1] = torch.sin(a) / s, torch.cos(a) / s
    theta[:, 2, 2], theta[:, :, 3] = 1 / s, t
    grid = F.affine_grid(theta, list(x.shape), align_corners=False)
    v = F.grid_sample(x, grid, padding_mode="zeros", align_corners=False)
    v = wedge_safe_flip(v, gen)
    v = v * (0.8 + 0.4 * torch.rand(B, 1, 1, 1, 1, generator=gen)) + 0.5 * torch.randn(v.shape, generator=gen)
    return (v - v.mean(dim=(2, 3, 4), keepdim=True)) / (v.std(dim=(2, 3, 4), keepdim=True) + 1e-6)


def nt_xent(z1, z2, tau=0.2):
    """SimCLR loss for a batch of positive pairs (z1[i], z2[i])."""
    z = F.normalize(torch.cat([z1, z2]), dim=1)
    sim = z @ z.T / tau
    sim.fill_diagonal_(float("-inf"))                     # a view is not its own partner
    B = len(z1)
    partner = torch.cat([torch.arange(B, 2 * B), torch.arange(0, B)]).to(z.device)
    return F.cross_entropy(sim, partner)


gen_demo = torch.Generator().manual_seed(1)
x_demo = torch.as_tensor(X_pool[:1]).repeat(4, 1, 1, 1, 1)
views = random_views(x_demo, gen_demo)
fig, axes = plt.subplots(1, 5, figsize=(8, 1.9))
axes[0].imshow(X_pool[0, 0].sum(0)); axes[0].set_title("original (projection)", fontsize=8)
for j in range(4):
    axes[j + 1].imshow(views[j, 0].sum(0).numpy()); axes[j + 1].set_title(f"view {j + 1}", fontsize=8)
for ax in axes:
    ax.axis("off")
fig.suptitle(f"Four random views of one {K_NAMES[y_pool[0]]} subtomogram: the network must learn they are 'the same'",
             fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
# ── 2.2 SimCLR pre-training on the unlabelled pool ──
class SimCLRNet(nn.Module):
    """Encoder f (64-d features) + projection head g (64 → 32) used only for the contrastive loss."""

    def __init__(self):
        super().__init__()
        self.encoder = Encoder3D()
        self.proj = nn.Sequential(nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 32))

    def forward(self, x):
        return self.proj(self.encoder(x))


def train_simclr(model, X, epochs=6, batch=32, lr=2e-3, tau=0.2, seed=SEED):
    """Contrastive pre-training: two random views per subtomogram, NT-Xent loss, no labels."""
    torch.manual_seed(seed); gen = torch.Generator().manual_seed(seed)
    model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs * (len(X) // batch))
    Xt, t0, hist = torch.as_tensor(X), time.time(), []
    for ep in range(epochs):
        model.train(); perm = torch.randperm(len(X), generator=gen); total = 0.0
        for i in range(0, len(X) - batch + 1, batch):
            xb = Xt[perm[i:i + batch]]
            loss = nt_xent(model(random_views(xb, gen).to(DEVICE)), model(random_views(xb, gen).to(DEVICE)), tau)
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
            total += loss.item()
        hist.append(total / (len(X) // batch))
        print(f"  epoch {ep + 1}/{epochs}  NT-Xent {hist[-1]:.3f}  (chance level ≈ ln(2B−1) = {math.log(2 * batch - 1):.2f})"
              f"  ({time.time() - t0:.0f} s)")
    return model


torch.manual_seed(SEED)
simclr = train_simclr(SimCLRNet(), X_pool, epochs=6)

### 2.2 The linear probe: how much is a label worth?

To judge features, the standard test is the **linear probe**: freeze the encoder, and train only a linear classifier
(logistic regression) on top, with $n$ labels per class. We compare four feature extractors, each repeated over 10
random draws of the labelled examples:

- a **random**, untrained encoder (a surprisingly strong baseline — random convolutions still measure size and texture);
- the **autoencoder** codes from Section 1;
- the **SimCLR** encoder;
- for reference, a fully supervised encoder is *not* allowed here — it would have used all labels.

In [ ]:
# ── 2.3 Linear probes with 1, 3 and 10 labels per class ──
torch.manual_seed(SEED + 1)
random_encoder = Encoder3D().to(DEVICE)
feature_sets = {"random encoder": random_encoder, "autoencoder codes": _Enc(ae, ae.encode),
                "SimCLR encoder": simclr.encoder}
FEATS = {name: (embed(m, X_pool), embed(m, X_test)) for name, m in feature_sets.items()}


def linear_probe(F_pool, F_test, n_per_class, draws=10, seed=SEED):
    """Mean ± sd test accuracy of logistic regression trained on n labelled examples per class."""
    r = np.random.default_rng(seed)
    mu, sd = F_pool.mean(0), F_pool.std(0) + 1e-6
    accs = []
    for _ in range(draws):
        idx = np.concatenate([r.choice(np.where(y_pool == k)[0], n_per_class, replace=False) for k in range(len(KNOWN))])
        clf = LogisticRegression(max_iter=2000, C=1.0).fit((F_pool[idx] - mu) / sd, y_pool[idx])
        accs.append(clf.score((F_test - mu) / sd, y_test))
    return np.mean(accs), np.std(accs)


SHOTS = [1, 3, 10]
probe = {name: [linear_probe(*FEATS[name], n) for n in SHOTS] for name in FEATS}
fig, ax = plt.subplots(figsize=(5.6, 3.4))
for name, res in probe.items():
    m, s = np.array(res).T
    ax.errorbar(SHOTS, m, yerr=s, marker="o", capsize=3, label=name)
    print(f"{name:18s}: " + "  ".join(f"{n:2d}/class → {a:.2f} ± {e:.2f}" for n, (a, e) in zip(SHOTS, res)))
ax.axhline(1 / len(KNOWN), color="gray", ls=":", label="chance")
ax.set_xscale("log"); ax.set_xticks(SHOTS); ax.set_xticklabels(SHOTS)
ax.set_xlabel("labelled subtomograms per class"); ax.set_ylabel("test accuracy (linear probe)")
ax.set_ylim(0, 1); ax.legend(fontsize=8); ax.set_title("What are unlabelled data worth?")
plt.tight_layout(); plt.show()

🔍 **What just happened?** With the encoder frozen, accuracy depends only on how well the features *already* separate
the molecules. Contrastive pre-training on 720 **unlabelled** subtomograms typically gives the best features at every
label budget, because its augmentations taught the encoder to ignore pose and noise; the autoencoder codes, which had
to remember pose, separate the classes less well. Notice also the random encoder: never dismiss a trivial baseline.
Gupta *et al.* found MoCo usually best among three contrastive methods; on real data with 25 % labels, pre-training
lifted accuracy from 46.5 % to 98.4 % (see box).

✏️ **Your turn.** In `random_views`, set `max_deg=0` (no rotations) and re-run 2.2–2.3. Which label budget suffers
most? What does that tell you about the role of augmentations?

### 📄 Paper connection — self-supervised learning for structure classification (Gupta *et al.* 2022)

**Paper**: Gupta T, He X, Uddin MR, Zeng X, Zhou A, Zhang J, Freyberg Z, Xu M. *Self-supervised learning for
macromolecular structure classification based on cryo-electron tomograms.* Front Physiol 13:957484 (2022).
[doi:10.3389/fphys.2022.957484](https://doi.org/10.3389/fphys.2022.957484)

- **Question it answers:** can unlabelled subtomograms pre-train an encoder so that few labels suffice?
- **What the paper actually does:** compares three contrastive self-supervised methods — **SimCLR, MoCo and SwAV** — on
  an RB3D (3D residual) backbone, with cryo-ET-specific augmentations each applied with p = 0.5 (random resized crop,
  scale 0.5–1; random 3D affine transform: rotation of ±45° about z, translation and scaling). MoCo: queue 128 (64 for
  real data), momentum 0.999, τ = 0.2, Adam lr 1e-4, 200 epochs; InfoNCE loss with a momentum-updated key encoder.
  Fine-tuning with 25–100 % of the labels (SGD, cosine schedule, lr 5e-4, 50 epochs).
- **What our toy keeps:** a contrastive objective (SimCLR's NT-Xent), the z-rotation/translation/scaling
  augmentations, a few-label evaluation.
- **What our toy leaves out:** MoCo's queue and momentum encoder, SwAV, the RB3D backbone, fine-tuning (we use a
  frozen linear probe), 200 epochs.
- **Result reported:** MoCo was usually best; simulated SNR 0.05 with 100 % labels: 67.3 % vs 47.9 % from random
  initialisation; real data (Noble *et al.*, 7 classes) with 25 % labels: 98.4 % vs 46.5 %.
- **Code:** none given in the article.

---
## 3 · Few-shot learning: recognising a new molecule from 1–5 examples

🧫 You have labelled many ribosomes and proteasomes (the **base classes**). Now five **novel** molecules appear — ClpB,
p97, a mitoribosome, TRiC and Hsp90 — with only **K = 1 or 5** labelled examples each (the **support set**), and you
must classify the rest (the **query set**). This is an **N-way K-shot** task; we evaluate 5-way tasks on the novel
molecules, averaged over many random tasks ("episodes").

### 3.1 Prototypical networks (Li R *et al.* 2020)

A **prototype** is the mean embedding of a class's support examples; a query is assigned to the nearest prototype:

$$\mathbf c_k = \frac{1}{|S_k|}\sum_{x \in S_k} f_\phi(x), \qquad
p(y = k \mid x) = \frac{\exp\!\big(-d(f_\phi(x), \mathbf c_k)\big)}{\sum_{k'} \exp\!\big(-d(f_\phi(x), \mathbf c_{k'})\big)},$$

with $d$ the squared Euclidean distance. *In words:* "which known group does this new particle sit closest to?" The key
idea is **episodic training**: during training on the base classes, every step *simulates* a few-shot task (sample N
classes, K support and some query examples, compute prototypes, minimise $-\log p(y = k^\ast \mid x)$ on the queries),
so the encoder learns an embedding in which "nearest prototype" works — for any classes.

In [ ]:
# ── 3.1 Episodic training of a prototypical network on the six base classes ──
def proto_loss(emb_s, emb_q, n_way, k_shot, n_query):
    """Prototypical loss and accuracy for one episode (support/query embeddings ordered class by class)."""
    protos = emb_s.view(n_way, k_shot, -1).mean(1)
    d = torch.cdist(emb_q, protos) ** 2                      # squared Euclidean distances [n_way·n_query, n_way]
    target = torch.arange(n_way).repeat_interleave(n_query).to(emb_q.device)
    return F.cross_entropy(-d, target), (d.argmin(1) == target).float().mean().item()


def train_protonet(encoder, X, y, episodes=120, n_way=5, k_shot=5, n_query=5, lr=2e-3, seed=SEED):
    """Each step: sample n_way base classes, k_shot support + n_query query boxes per class, minimise proto_loss."""
    torch.manual_seed(seed); gen = torch.Generator().manual_seed(seed); r = np.random.default_rng(seed)
    encoder.to(DEVICE)
    opt = torch.optim.Adam(encoder.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=episodes)
    t0, accs = time.time(), []
    for ep in range(episodes):
        encoder.train()
        classes = r.choice(np.unique(y), n_way, replace=False)
        picks = [r.choice(np.where(y == c)[0], k_shot + n_query, replace=False) for c in classes]
        s_idx = np.concatenate([p[:k_shot] for p in picks]); q_idx = np.concatenate([p[k_shot:] for p in picks])
        xb = wedge_safe_flip(torch.as_tensor(np.concatenate([X[s_idx], X[q_idx]])), gen).to(DEVICE)
        emb = encoder(xb)
        loss, acc = proto_loss(emb[:len(s_idx)], emb[len(s_idx):], n_way, k_shot, n_query)
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        accs.append(acc)
        if (ep + 1) % 30 == 0:
            print(f"  episode {ep + 1}/{episodes}: mean training-episode accuracy (last 30) {np.mean(accs[-30:]):.2f}"
                  f"  ({time.time() - t0:.0f} s)")
    return encoder


torch.manual_seed(SEED)
protonet = train_protonet(Encoder3D(), X_pool, y_pool)

### 3.2 Testing on molecules never seen in training

We compare encoders on the **novel** molecules using the same nearest-prototype rule (on L2-normalised features):
random, SimCLR (no labels), a **supervised** encoder trained with ordinary cross-entropy on the base classes (the
"Baseline" of the few-shot literature — we train it now and reuse it in Section 5 as the softmax open-set baseline),
and the ProtoNet encoder.

In [ ]:
# ── 3.2a A supervised base-class encoder (plain softmax classifier) ──
class SoftmaxNet(nn.Module):
    """Encoder → 32-d embedding → linear layer to the 6 known classes (ordinary softmax classifier)."""

    def __init__(self, n_classes=6, emb_dim=32):
        super().__init__()
        self.encoder = Encoder3D()
        self.embed = nn.Linear(64, emb_dim)
        self.fc = nn.Linear(emb_dim, n_classes)

    def embedding(self, x):
        return self.embed(self.encoder(x))

    def forward(self, x):
        return self.fc(F.relu(self.embedding(x)))


def train_supervised(model, X, y, epochs=6, batch=32, lr=2e-3, extra_loss=None, seed=SEED):
    """Cross-entropy training with wedge-safe flips; `extra_loss(model, xb, yb)` may add terms (Section 5)."""
    torch.manual_seed(seed); gen = torch.Generator().manual_seed(seed)
    model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs * math.ceil(len(X) / batch))
    Xt, yt, t0 = torch.as_tensor(X), torch.as_tensor(y), time.time()
    for ep in range(epochs):
        model.train(); perm = torch.randperm(len(X), generator=gen); total = 0.0
        for i in range(0, len(X), batch):
            idx = perm[i:i + batch]
            xb, yb = wedge_safe_flip(Xt[idx], gen).to(DEVICE), yt[idx].to(DEVICE)
            loss = F.cross_entropy(model(xb), yb) + (extra_loss(model, xb, yb) if extra_loss else 0.0)
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
            total += loss.item() * len(idx)
        print(f"  epoch {ep + 1}/{epochs}  loss {total / len(X):.3f}  ({time.time() - t0:.0f} s)")
    return model


torch.manual_seed(SEED)
softmax_net = train_supervised(SoftmaxNet(), X_pool, y_pool, epochs=6)
with torch.no_grad():
    acc_known = (embed(softmax_net, X_test).argmax(1) == y_test).mean()
print(f"softmax classifier on the 6 known classes: test accuracy {acc_known:.2f}")

In [ ]:
# ── 3.2b 5-way episodes on the NOVEL molecules ──
def l2n(F_):
    """L2-normalise feature vectors (rows)."""
    return F_ / (np.linalg.norm(F_, axis=1, keepdims=True) + 1e-9)


def few_shot_eval(F_nov, k_shot, classify, episodes=300, n_query=10, seed=SEED):
    """Mean accuracy (± 95 % CI) over random 5-way k-shot episodes drawn from the novel set."""
    r = np.random.default_rng(seed)
    accs = []
    for _ in range(episodes):
        picks = [r.choice(np.where(y_nov == c)[0], k_shot + n_query, replace=False) for c in range(len(NOVEL))]
        s = np.concatenate([p[:k_shot] for p in picks]); q = np.concatenate([p[k_shot:] for p in picks])
        ys = np.repeat(np.arange(len(NOVEL)), k_shot); yq = np.repeat(np.arange(len(NOVEL)), n_query)
        accs.append((classify(F_nov[s], ys, F_nov[q]) == yq).mean())
    return float(np.mean(accs)), float(1.96 * np.std(accs) / np.sqrt(episodes))


def nearest_prototype(Fs, ys, Fq):
    """Assign each query to the class whose mean (prototype) is nearest (on L2-normalised features)."""
    Fs, Fq = l2n(Fs), l2n(Fq)
    protos = np.stack([Fs[ys == c].mean(0) for c in np.unique(ys)])
    return np.argmin(((Fq[:, None] - protos[None]) ** 2).sum(-1), axis=1)


encoders = {"random encoder": random_encoder, "SimCLR (no labels)": simclr.encoder,
            "supervised softmax": _Enc(softmax_net, softmax_net.encoder), "ProtoNet (episodic)": protonet}
F_NOV = {name: embed(m, X_nov) for name, m in encoders.items()}
F_BASE = {name: embed(m, X_pool) for name, m in encoders.items()}
fs = {}
for name in encoders:
    fs[name] = [few_shot_eval(F_NOV[name], k, nearest_prototype) for k in (1, 5)]
    print(f"{name:20s}: 5-way 1-shot {fs[name][0][0]:.2f} ± {fs[name][0][1]:.2f} | "
          f"5-shot {fs[name][1][0]:.2f} ± {fs[name][1][1]:.2f}   (chance 0.20)")

🔍 **What just happened?** None of these encoders ever saw ClpB, p97, the mitoribosome, TRiC or Hsp90, yet with one
example per class they classify them well above chance (0.20), and five examples help a lot. Encoders trained on the
base classes (softmax or episodic) learned features — size, elongation, rings, two-lobed shapes — that transfer to new
molecules. Li *et al.* found ProtoNet-style training competitive or better than such baselines, especially with an
extra task-adaptation module (ProtoNet-CE). With our tiny budget (120 training episodes) the ranking between the trained
encoders can flip; the gap to the random encoder is the robust result.

### 3.3 FSCC: borrowing statistics from the base classes (Gao *et al.* 2022)

With one example, a prototype is just that example — noise included. **Distribution calibration** (Yang *et al.* 2021,
used by FSCC) assumes that a novel class's feature distribution resembles those of **similar base classes**, whose
means $\mu_i$ and covariances $\Sigma_i$ we can estimate from plenty of data:

1. **Tukey transform** $\tilde x = x^{\lambda}$ with $\lambda = 0.5$ (a square root) makes skewed, non-negative features
   more Gaussian;
2. for each support feature $\tilde x$, find the $k$ nearest base-class means and **calibrate**:
   $$\mu' = \frac{\sum_{i \in \text{top-}k} \mu_i + \tilde x}{k + 1}, \qquad \Sigma' = \frac{1}{k}\sum_{i \in \text{top-}k} \Sigma_i + \alpha I ;$$
3. **sample** many synthetic features from $\mathcal N(\mu', \Sigma')$ and train a logistic-regression classifier on the
   support plus the samples.

*In words:* "a new two-lobed molecule probably varies the way ribosomes vary". (These equations follow the original
distribution-calibration method; FSCC's exact equations did not render in the source we could read, and $k$ is not
stated there — we use $k = 2$.)

In [ ]:
# ── 3.3 Distribution calibration on top of a frozen encoder ──
def tukey(F_, lam=0.5):
    """Tukey's ladder of powers (λ = 0.5 → square root) for non-negative features."""
    return np.power(np.clip(F_, 0, None), lam)


def make_calibrated_classifier(F_base, k=2, n_per_class=100, alpha_frac=0.2, borrow_mean=True, seed=SEED):
    """Return classify(Fs, ys, Fq): logistic regression on the support plus features sampled from calibrated Gaussians.

    borrow_mean=True  → μ' = (Σ top-k base means + x̃)/(k + 1)   (distribution calibration as published)
    borrow_mean=False → μ' = x̃                                (borrow only the spread Σ' from the base classes)
    """
    Fb = tukey(F_base)
    mus = np.stack([Fb[y_pool == c].mean(0) for c in range(len(KNOWN))])
    covs = np.stack([np.cov(Fb[y_pool == c].T) for c in range(len(KNOWN))])
    alpha = alpha_frac * np.mean([np.trace(c) / len(c) for c in covs])
    # Σ' depends only on WHICH k base classes are nearest → pre-compute one Cholesky factor per combination
    chol = {combo: np.linalg.cholesky(covs[list(combo)].mean(0) + alpha * np.eye(Fb.shape[1]))
            for combo in combinations(range(len(KNOWN)), k)}
    r = np.random.default_rng(seed)

    def classify(Fs, ys, Fq):
        Fs, Fq = tukey(Fs), tukey(Fq)
        per_support = max(1, n_per_class // np.bincount(ys).max())
        X_aug, y_aug = [Fs], [ys]
        for x, yv in zip(Fs, ys):
            near = tuple(sorted(np.argsort(((mus - x) ** 2).sum(1))[:k]))
            mu = (mus[list(near)].sum(0) + x) / (k + 1) if borrow_mean else x
            X_aug.append(mu + r.standard_normal((per_support, len(x))) @ chol[near].T)
            y_aug.append(np.full(per_support, yv))
        clf = LogisticRegression(max_iter=500).fit(np.concatenate(X_aug), np.concatenate(y_aug))
        return clf.predict(Fq)
    return classify


def support_only_lr(Fs, ys, Fq):
    """Logistic regression on the (Tukey-transformed) support features only."""
    return LogisticRegression(max_iter=500).fit(tukey(Fs), ys).predict(tukey(Fq))


t0 = time.time()
dc_rows = {}
name = "supervised softmax"                       # FSCC calibrates a supervised backbone
for label, fn in [("nearest prototype", nearest_prototype), ("LR on support only", support_only_lr),
                  ("LR + calibration (mean & spread)", make_calibrated_classifier(F_BASE[name])),
                  ("LR + calibration (spread only)", make_calibrated_classifier(F_BASE[name], borrow_mean=False))]:
    dc_rows[label] = [few_shot_eval(F_NOV[name], k, fn, episodes=60) for k in (1, 5)]
    print(f"{label:32s}: 5-way 1-shot {dc_rows[label][0][0]:.2f} ± {dc_rows[label][0][1]:.2f} | "
          f"5-shot {dc_rows[label][1][0]:.2f} ± {dc_rows[label][1][1]:.2f}")
print(f"(features: {name}; 60 episodes per setting; {time.time() - t0:.0f} s)")

fig, ax = plt.subplots(figsize=(6, 3))
for j, (lab, res) in enumerate(dc_rows.items()):
    ax.bar(np.arange(2) + (j - 1.5) * 0.2, [r[0] for r in res], 0.2, yerr=[r[1] for r in res], capsize=3, label=lab)
ax.set_xticks(range(2)); ax.set_xticklabels(["5-way 1-shot", "5-way 5-shot"])
ax.axhline(0.2, color="gray", ls=":"); ax.set_ylim(0, 1); ax.set_ylabel("accuracy on novel molecules")
ax.set_title("Few-shot classifiers on frozen supervised features (±95 % CI)", fontsize=9)
ax.legend(fontsize=7, loc="upper left")
plt.tight_layout(); plt.show()

🔍 **What just happened?** A logistic regression trained on a single example per class is a shaky classifier;
adding synthetic features sampled around the calibrated mean gives it a sense of how much each class *varies*, which
usually helps most in the 1-shot setting. How much it helps depends on whether the base classes really resemble the
novel ones — calibration borrows the *spread* of ribosomes and proteasomes, which suits a mitoribosome better than a
small chaperone. FSCC reports gains of about 6–17 points on real data (box below).

✏️ **Your turn.** Change `k=2` to `k=1` or `k=4` in `make_calibrated_classifier` and `alpha_frac` to 0 or 1. When does
borrowing from more base classes hurt?

### 📄 Paper connection — few-shot learning (Li R *et al.* 2020; Gao *et al.* 2022)

**Paper 1**: Li R, Yu L, Zhou B, Zeng X, Wang Z, Yang X, Zhang J, Gao X, Jiang R, Xu M. *Few-shot learning for
classification of novel macromolecular structures in cryo-electron tomograms.* PLOS Comput Biol 16(11):e1008227
(2020). [doi:10.1371/journal.pcbi.1008227](https://doi.org/10.1371/journal.pcbi.1008227)

- **What the paper actually does:** **ProtoNet3D** — embedding of 4 ConvBlocks (64 × 3³ conv → BatchNorm → ReLU →
  2³ max-pool), class prototypes = mean support embeddings, softmax over negative squared Euclidean distances, episodic
  training (e.g. 5-way 5-shot with 15 queries per class; Adam lr 1e-4; mixed-SNR training). **ProtoNet-CE**
  (**Combined Embedding**, not cross-entropy) adds a FEAT-style self-attention module that adapts embeddings to each task,
  trained in a second stage with the encoder frozen; at test time task-agnostic and task-specific distances are summed.
  Data: simulated 40³ subtomograms (22 complexes + NULL; 10 training, 5 validation, 8 test classes) and two real
  datasets.
- **What our toy keeps:** ProtoNet3D's prototypes, distance and episodic training; evaluation on unseen classes.
- **What our toy leaves out:** ProtoNet-CE's attention module, mixed-SNR training, the real datasets, 64-channel blocks.
- **Result reported:** simulated SNR 0.1: > 0.86 at 5-shot and 0.7644 at 1-shot; rat-neuron data: ProtoNet-CE 0.9406
  (5-shot) and 0.9153 (1-shot) vs 0.9227 and 0.8407 for ProtoNet3D; Noble single-particle data: both 100 %.
- **Code:** AITom `aitom/classify/deep/supervised/cnn/few_shot/protonet/`.

**Paper 2**: Gao S, Zeng X, Xu M, Zhang F. *FSCC: Few-shot learning for macromolecule classification based on contrastive
learning and distribution calibration in cryo-electron tomography.* Front Mol Biosci 9:931949 (2022).
[doi:10.3389/fmolb.2022.931949](https://doi.org/10.3389/fmolb.2022.931949)

- **What the paper actually does:** stage 1 — a Conv-6 3D backbone + FC classifier trained on base classes with
  $L = L_{CE} + \lambda L_{SCL}$ ($\lambda = 0.05$; $L_{SCL}$ = supervised contrastive loss of Khosla *et al.* with
  τ = 0.07; class-balanced batches); stage 2 — freeze the backbone, compute base-class means and covariances, apply the
  Tukey transform (λ = 0.5) to novel features, calibrate each support feature's distribution from its nearest base
  classes, sample synthetic features and retrain the classifier.
- **What our toy keeps:** frozen backbone, Tukey transform, nearest-base-class calibration, sampling, logistic regression.
- **What our toy leaves out:** the supervised contrastive term (our backbones are plain softmax or SimCLR), Conv-6, the
  SHREC data. ⚠️ The calibration equations above are those of the original method (Yang *et al.* 2021); FSCC's own
  Eqs. 9–10 did not render in the version we read, and its $k$ was not stated.
- **Result reported:** SHREC'19 5-way 5-shot 77.0 % vs 75.3 % for Baseline++ (which uses 50 labels per class); real
  data improvements of about 6–17 points over the baseline (abstract: 5–16 %), e.g. Dataset1 3-way 5-shot 86.8 % vs 70.0 %.
- **Code:** none in the article or AITom.

---
## 4 · Active learning: which particles should the expert label next?

🧫 Labelling is the bottleneck, so *which* particles we label matters. **Active learning** runs a loop: train on the
current labelled set → score every unlabelled particle → ask the expert to label the top-scoring batch → repeat.
Scores from the HAL paper (Du *et al.* 2021):

- **Uncertainty** — the predictive **entropy** $E(x) = -\sum_y P(y\mid x)\log P(y\mid x)$: label what the model is
  unsure about.
- **Representativeness** — train a binary **discriminator** $D$ on the classifier's intermediate features $M(x)$ to
  tell labelled (0) from unlabelled (1) particles; a high $\Pr(D(M(x)) = 1)$ means "this region of feature space is not
  yet covered by labels".
- **HAL (hybrid)**: $S(x) = \Pr\big(D(M(x)) = 1\big) + \lambda E(x)$ with $\lambda = 1$, plus **subset sampling**: pick
  the batch in $m$ sub-batches, retraining $D$ after each, so the picks do not all come from the same region.

To fit a CPU budget, our classifier is a small network on top of the **frozen SimCLR features** from Section 2 (a
realistic modern pipeline: self-supervised features first, labels later); HAL retrains the full CNN in every round. We
start with 12 random labels (≈ 1.7 % of the pool) and query 24 per round.

In [ ]:
# ── 4.1 The active-learning loop: random vs entropy vs HAL ──
F_al_pool = torch.as_tensor(FEATS["SimCLR encoder"][0]); F_al_test = torch.as_tensor(FEATS["SimCLR encoder"][1])
mu_al, sd_al = F_al_pool.mean(0), F_al_pool.std(0) + 1e-6
F_al_pool, F_al_test = (F_al_pool - mu_al) / sd_al, (F_al_test - mu_al) / sd_al


class SmallClassifier(nn.Module):
    """64-d features → hidden layer M(x) (32-d) → 6 classes."""

    def __init__(self):
        super().__init__()
        self.hidden = nn.Sequential(nn.Linear(64, 32), nn.ReLU())
        self.out = nn.Linear(32, len(KNOWN))

    def forward(self, f):
        return self.out(self.hidden(f))


def fit(model, Xf, y, steps=100, lr=1e-2, seed=0):
    """Full-batch Adam training (cross-entropy or binary cross-entropy for 1-output models)."""
    torch.manual_seed(seed)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-3)
    yt = torch.as_tensor(y)
    for _ in range(steps):
        out = model(Xf)
        loss = F.binary_cross_entropy_with_logits(out[:, 0], yt.float()) if out.shape[1] == 1 else F.cross_entropy(out, yt)
        opt.zero_grad(); loss.backward(); opt.step()
    return model


def query(strategy, clf, labelled, budget, r, m_sub=3, lam=1.0):
    """Choose `budget` new pool indices to label."""
    unl = np.setdiff1d(np.arange(len(y_pool)), labelled)
    if strategy == "random":
        return r.choice(unl, budget, replace=False)
    with torch.no_grad():
        p = F.softmax(clf(F_al_pool), 1).numpy()
        M = clf.hidden(F_al_pool)
    entropy = -(p * np.log(p + 1e-10)).sum(1)
    if strategy == "entropy":
        return unl[np.argsort(-entropy[unl])[:budget]]
    picked, lab = [], list(labelled)                              # HAL: discriminator + entropy, in sub-batches
    for sub in range(m_sub):
        unl = np.setdiff1d(np.arange(len(y_pool)), lab)
        disc = fit(nn.Sequential(nn.Linear(32, 32), nn.ReLU(), nn.Linear(32, 1)),
                   torch.cat([M[np.array(lab)], M[unl]]), np.r_[np.zeros(len(lab)), np.ones(len(unl))].astype(int),
                   steps=60, seed=sub)
        with torch.no_grad():
            p_unl = torch.sigmoid(disc(M[unl])[:, 0]).numpy()
        score = p_unl + lam * entropy[unl]
        n = budget // m_sub + (1 if sub < budget % m_sub else 0)
        new = unl[np.argsort(-score)[:n]]
        picked.extend(new); lab.extend(new)
    return np.array(picked)


def active_learning(strategy, seed, n_start=12, budget=24, rounds=7):
    """Run the loop; return (labels used, test accuracy) after each round."""
    r = np.random.default_rng(seed)
    labelled = r.choice(len(y_pool), n_start, replace=False)
    curve = []
    for rd in range(rounds + 1):
        clf = fit(SmallClassifier(), F_al_pool[labelled], y_pool[labelled], seed=seed)
        with torch.no_grad():
            acc = (clf(F_al_test).argmax(1).numpy() == y_test).mean()
        curve.append((len(labelled), acc))
        if rd < rounds:
            labelled = np.concatenate([labelled, query(strategy, clf, labelled, budget, r)])
    return np.array(curve)


t0 = time.time()
AL = {s: np.stack([active_learning(s, seed) for seed in range(3)]) for s in ["random", "entropy", "HAL (disc. + entropy)"]}
full = fit(SmallClassifier(), F_al_pool, y_pool)
with torch.no_grad():
    acc_full = (full(F_al_test).argmax(1).numpy() == y_test).mean()
print(f"active learning: 3 strategies × 3 seeds in {time.time() - t0:.0f} s; all {len(y_pool)} labels → {acc_full:.2f}")

fig, ax = plt.subplots(figsize=(5.8, 3.4))
for s, curves in AL.items():
    n, acc = curves[0, :, 0], curves[:, :, 1]
    ax.plot(n, acc.mean(0), "o-", label=s)
    ax.fill_between(n, acc.mean(0) - acc.std(0), acc.mean(0) + acc.std(0), alpha=0.15)
    print(f"{s:22s}: accuracy with {int(n[-1])} labels = {acc[:, -1].mean():.2f} ± {acc[:, -1].std():.2f}")
ax.axhline(acc_full, color="k", ls="--", label=f"all {len(y_pool)} labels")
ax.set_xlabel("number of labelled subtomograms"); ax.set_ylabel("test accuracy")
ax.set_title("Active learning on frozen self-supervised features (mean ± sd, 3 seeds)"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

🔍 **What just happened?** All three strategies start from the same 12 random labels. Querying by uncertainty and by
HAL's hybrid score usually climbs faster than random sampling, approaching the all-labels accuracy with a fraction of
the labels — the effect HAL reports at much larger scale. Pure entropy can get stuck asking about the same confusing
region (e.g. 70S vs 80S); the discriminator term and sub-batches push the queries to spread out. With small pools and
three seeds the curves overlap at places: **active learning gains are real but modest**, and they depend on the
quality of the features.

✏️ **Your turn.** Set `lam=0` in `query` (discriminator only) or `m_sub=1` (no subset sampling) and re-run 4.1. Which
ingredient matters most here?

### 📄 Paper connection — Hybrid Active Learning, HAL (Du *et al.* 2021)

**Paper**: Du X, Wang H, Zhu Z, Zeng X, Chang YW, Zhang J, Xing E, Xu M. *Active learning to classify macromolecular
structures in situ for less supervision in cryo-electron tomography.* Bioinformatics (2021).
doi:10.1093/bioinformatics/btab123 (volume/pages not verified) · [arXiv:2102.12040](https://arxiv.org/abs/2102.12040)

- **Question it answers:** which subtomograms should be annotated so that a classifier reaches high accuracy with far
  fewer labels?
- **What the paper actually does:** each round trains the CNN classifier (softmax cross-entropy), trains a binary
  discriminator on intermediate features to separate labelled from unlabelled samples (representativeness), scores
  uncertainty by entropy, ranks by $S = \Pr(D(M(x)) = 1) + \lambda E$ ($\lambda = 1$), and fills the query batch in
  sub-batches, retraining $D$ in between. A bound links the unlabelled error to the labelled error plus a domain
  divergence that the discriminator estimates. The paper states that HAL involves **no adversarial training**. Data:
  simulated (23 classes × 1,000, SNR 0.03/0.05) and real (rat neuron; EMPIAR/Noble data); initial labelled set 3–4 %.
- **What our toy keeps:** entropy + discriminator score, λ = 1, sub-batches, comparison with random and entropy-only.
- **What our toy leaves out:** retraining the CNN each round (we use frozen self-supervised features), the large pools
  and query batches (800 simulated / 32 real), the BADGE and other baselines.
- **Result reported:** using 9–24 % of the labels, HAL came within about 1.4–6.7 points of fully supervised accuracy
  (e.g. S2 93.96 vs 95.36; R2 95.0 vs 99.0) and beat BADGE and other active-learning baselines on every dataset.
- **Code:** AITom `aitom/classify/deep/supervised/cnn/active/hal/` (`query_methods.py`: score = discriminator
  probability + `lambda_e` × entropy, with sub-batches).

**Related work from the lab (cited only):**
- Wang T, Li B, Zhang J, Zeng X, Uddin MR, Wu W, Xu M. *Deep active learning for cryo-electron tomography
  classification.* IEEE ICIP 2022. ⚠️ Content not verified — we could not access this paper; do not assume which
  acquisition criterion it uses.
- Wang T, Li X, Yang P, Hu G, Zeng X, Huang S, Xu C-Z, Xu M. *Boosting active learning via improving test performance*
  (AL-GradNorm), AAAI 2022 — a general (not cryo-ET-specific) method; its README states that, via the influence
  function, unlabelled data with a higher **gradient norm** should be selected. Code: `aitom/ml/active_learning/al_gradnorm/`.
- Li X, Yang P, Gu Y, Zhan X, Wang T, Xu M, Xu C. *Deep active learning with noise stability.* AAAI 2024,
  38(12):13655–13663 — general active learning; content not verified for this course.

---
## 5 · Open-set recognition: "none of the above" (Du *et al.* 2019)

🧫 A classifier trained on six molecules will *always* answer with one of six names — even for a ClpB or a p97 it has
never seen. In a cell, unseen molecules are the rule, not the exception. **Open-set recognition** asks the network to
also say "unknown". The trick of Du *et al.*: learn an **embedding** in which each class forms a tight, well-separated
cluster, so that anything far from every cluster can be flagged — and two particles of an *unseen* class still land
close to each other.

**Large Margin Cosine Loss (LMCL).** Normalise the embedding $x_i$ and the class weight vectors $W_j$, so the logit is
a cosine $\cos\theta_j$; subtract a margin $m$ from the true class and scale by $s$:

$$L_{LMCL} = -\frac{1}{N}\sum_i \log\frac{e^{s(\cos\theta_{y_i} - m)}}{e^{s(\cos\theta_{y_i} - m)} + \sum_{j \neq y_i} e^{s\cos\theta_j}} .$$

*In words:* the correct class must win by a margin of $m$ in cosine similarity — classes are pushed apart in angle.

**Center loss.** $L_{Center} = \tfrac12\sum_i \lVert x_i - c_{y_i}\rVert^2$ pulls each embedding towards a running
centre of its class (centres updated by $c_j \leftarrow c_j - \alpha\,\Delta c_j$ with
$\Delta c_j = \frac{\sum_i \delta(y_i = j)(c_j - x_i)}{1 + \sum_i \delta(y_i = j)}$). The total loss of the paper is

$$L_{all} = L_{LMCL} + \lambda L_{Center} + L_{Softmax} + L_{reg}.$$

We train it next to the plain softmax network of Section 3 and test with **ClpB and p97 as unknowns**.

In [ ]:
# ── 5.1 Train the Soft Large Margin Centralized Cosine Loss network ──
class CosineMarginNet(SoftmaxNet):
    """SoftmaxNet + a cosine-margin head (normalised class weights) and class centres for the center loss."""

    def __init__(self, n_classes=6, emb_dim=32, s=16.0, m=0.35):
        super().__init__(n_classes, emb_dim)
        self.W = nn.Parameter(torch.randn(n_classes, emb_dim) * 0.1)
        self.register_buffer("centres", torch.zeros(n_classes, emb_dim))
        self.s, self.m = s, m

    def cosine_logits(self, x):
        return F.normalize(self.embedding(x), dim=1) @ F.normalize(self.W, dim=1).T


def soft_lmccl_terms(model, xb, yb, lam=0.1, alpha=0.5):
    """LMCL + λ·center loss (the softmax term is added by train_supervised); updates the centres in place."""
    emb = model.embedding(xb)
    cos = F.normalize(emb, dim=1) @ F.normalize(model.W, dim=1).T
    margin = model.m * F.one_hot(yb, cos.shape[1])
    lmcl = F.cross_entropy(model.s * (cos - margin), yb)
    centre_loss = 0.5 * ((emb - model.centres[yb]) ** 2).sum(1).mean()
    with torch.no_grad():                                     # the paper's centre update rule
        for j in yb.unique():
            sel = yb == j
            delta = (model.centres[j] - emb[sel]).sum(0) / (1 + sel.sum())
            model.centres[j] -= alpha * delta
    return lmcl + lam * centre_loss


torch.manual_seed(SEED)
lmcc_net = train_supervised(CosineMarginNet(), X_pool, y_pool, epochs=6, extra_loss=soft_lmccl_terms)
with torch.no_grad():
    acc_cos = (np.concatenate([lmcc_net.cosine_logits(torch.as_tensor(X_test[i:i + 128]).to(DEVICE)).cpu().numpy()
                               for i in range(0, len(X_test), 128)]).argmax(1) == y_test).mean()
print(f"known-class test accuracy: softmax net {acc_known:.2f} | cosine-margin + center net {acc_cos:.2f}")

In [ ]:
# ── 5.2 Detect unknown molecules by their distance to the known class centres ──
UNKNOWN = [NOVEL.index("1qvr"), NOVEL.index("3cf3")]                 # ClpB and p97
X_unk = X_nov[np.isin(y_nov, UNKNOWN)]
nets = {"softmax (Section 3)": softmax_net, "LMCL + center loss": lmcc_net}
osr = {}
fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.1))
for col, (name, net) in enumerate(nets.items()):
    emb = lambda X: l2n(embed(_Enc(net, net.embedding), X))
    E_pool, E_test, E_unk, E_nov = emb(X_pool), emb(X_test), emb(X_unk), emb(X_nov)
    centres = l2n(np.stack([E_pool[y_pool == c].mean(0) for c in range(len(KNOWN))]))
    dist = lambda E: np.sqrt(((E[:, None] - centres[None]) ** 2).sum(-1)).min(1)   # distance to nearest centre
    d_known, d_unk = dist(E_test), dist(E_unk)
    auroc = roc_auc_score(np.r_[np.zeros(len(d_known)), np.ones(len(d_unk))], np.r_[d_known, d_unk])
    thr = np.quantile(d_known, 0.95)                                          # keep 95 % of knowns
    flagged = (d_unk > thr).mean()
    # the paper's protocol: are two subtomograms of UNSEEN classes the same molecule? (all novel pairs)
    iu = np.triu_indices(len(E_nov), 1)
    pair_d = np.sqrt(((E_nov[iu[0]] - E_nov[iu[1]]) ** 2).sum(-1))
    same = (y_nov[iu[0]] == y_nov[iu[1]]).astype(int)
    pair_auc = roc_auc_score(same, -pair_d)
    osr[name] = dict(auroc=auroc, flagged=flagged, pair_auc=pair_auc)
    print(f"{name:20s}: unknown-vs-known AUROC {auroc:.2f} | ClpB/p97 flagged at 95 % known retention: "
          f"{100 * flagged:.0f} % | unseen-class pair verification AUC {pair_auc:.2f}")
    axes[col].hist(d_known, bins=30, alpha=0.6, density=True, label="known test molecules")
    axes[col].hist(d_unk, bins=30, alpha=0.6, density=True, label="ClpB + p97 (unknown)")
    axes[col].axvline(thr, color="k", ls="--", lw=1)
    axes[col].set_title(f"{name}\nAUROC {auroc:.2f}", fontsize=9)
    axes[col].set_xlabel("distance to nearest known centre"); axes[col].legend(fontsize=7)
    fpr, tpr, _ = roc_curve(np.r_[np.zeros(len(d_known)), np.ones(len(d_unk))], np.r_[d_known, d_unk])
    axes[2].plot(fpr, tpr, label=f"{name} ({auroc:.2f})")
axes[2].plot([0, 1], [0, 1], "k:", lw=1); axes[2].set_xlabel("false 'unknown' rate (knowns)")
axes[2].set_ylabel("detected unknowns"); axes[2].set_title("ROC: known vs unknown", fontsize=9); axes[2].legend(fontsize=7)
plt.tight_layout(); plt.show()

In [ ]:
# ── 5.3 Where do the unknowns land? A 2D view of the embeddings ──
fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))
for ax, (name, net) in zip(axes, nets.items()):
    E_test = l2n(embed(_Enc(net, net.embedding), X_test)); E_unk = l2n(embed(_Enc(net, net.embedding), X_unk))
    pca = PCA(n_components=2, random_state=SEED).fit(E_test)
    Z, Zu = pca.transform(E_test), pca.transform(E_unk)
    for k in range(len(KNOWN)):
        ax.scatter(*Z[y_test == k].T, s=6, color=plt.cm.tab10(k), label=K_NAMES[k])
    ax.scatter(*Zu.T, s=10, color="k", marker="x", label="ClpB / p97 (unknown)")
    ax.set_title(name, fontsize=9); ax.set_xticks([]); ax.set_yticks([])
axes[1].legend(fontsize=7, loc="center left", bbox_to_anchor=(1, 0.5), markerscale=2)
fig.suptitle("Normalised embeddings (PCA to 2D): known classes in colour, unknown molecules as black crosses", fontsize=9)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Both networks classify the six known molecules, but they organise the embedding
differently. The cosine margin and center loss are designed to make each known class a **tight, angularly separated**
cluster; in our toy this typically makes unknowns stand out more (higher AUROC) and keeps two copies of an *unseen*
molecule closer together (higher pair-verification AUC, the paper's own test). Look at the histograms: no threshold
separates knowns from unknowns perfectly — ClpB and p97 are hexameric AAA+ machines of roughly RuBisCO-like size,
and at SNR 0.1 and 1.5 nm voxels they genuinely resemble some known classes. Du *et al.* report verification accuracy
0.794 (AUC 0.862) on unseen simulated classes, and much lower on real data (0.602 / 0.671): open-set recognition in
cells remains hard.

✏️ **Your turn.** In `CosineMarginNet`, try `m=0.0` (no margin) or set `lam=0` in `soft_lmccl_terms` (no center loss).
Which term matters more for detecting unknowns? (The paper's ablation found the combination best.)

### 📄 Paper connection — open-set recognition with Soft LMCCL (Du *et al.* 2019)

**Paper**: Du X, Zeng X, Zhou B, Singh A, Xu M. *Open-set recognition of unseen macromolecules in cellular electron
cryo-tomograms by soft large margin centralized cosine loss.* BMVC 2019 (spotlight).
[PDF](https://bmvc2019.org/wp-content/uploads/papers/0347-paper.pdf)

- **Question it answers:** classifiers trained on known structures cannot handle new structures in situ — can we
  learn an embedding in which even unseen classes can be judged "same" or "different"?
- **What the paper actually does:** a 3D ResNet-20-like network trained on seen classes with
  $L_{all} = L_{LMCL} + \lambda L_{Center} + L_{Softmax} + L_{reg}$ ($s = 64$; $m = 0.5$ or $0.35$; $\lambda = 0.1$ or
  $0.01$; weight decay 5e-4); a 1,024-d FC embedding; **at test time no classification**: two subtomograms are embedded
  and declared different when their L2 distance exceeds a threshold chosen by K-fold cross-validation. The word "soft" is
  not formally defined. Data: simulated (23 classes, SNR 0.03/0.05; 17 seen, 6 unseen) and real rat-neuron
  subtomograms (6 classes; 3 seen, 3 unseen), plus cross-domain tests.
- **What our toy keeps:** LMCL + center loss (with the paper's centre update rule) + an auxiliary softmax, weight decay;
  unseen-class pair verification; plus an unknown-detection test by distance to known centres.
- **What our toy leaves out:** the ResNet, the 1,024-d embedding, K-fold threshold selection, cross-domain tests; we use
  $s = 16$ (better suited to our 32-d embedding) instead of 64.
- **Result reported:** Soft LMCCL beat softmax-only, center-only and LMCL-only training; S1 unseen classes: accuracy
  0.794, AUC 0.862; real R1: accuracy 0.602, AUC 0.671.
- **Code:** AITom `aitom/classify/deep/supervised/cnn/openset/` (TensorFlow; `train_double.py`:
  `loss = coco_loss + reg + center_weighting*center_loss + cross_entropy`).

**Related (cited only):** Zeng Y, Howe G, Zeng X, Zhang J, Chang YW, Xu M. *Unsupervised domain alignment based open set
structural recognition of macromolecules captured by cryo-electron tomography.* IEEE ICIP 2021. ⚠️ Content not
verified — the title suggests combining open-set recognition with unsupervised domain alignment (Module 18), but we
could not read the paper.

---
## 🧭 Which strategy when?

| your situation | try first | what it assumes |
|---|---|---|
| many particles, no labels yet | self-supervised pre-training (Section 2), then clustering (Module 20) | augmentations encode what should *not* matter |
| a few labels for every class | semi-supervised / linear probe on self-supervised features (Sections 1–2) | unlabelled data come from the same classes |
| many labels for some molecules, 1–5 for new ones | few-shot: prototypes + calibration (Section 3) | new classes resemble base classes in feature space |
| an expert available for a limited time | active learning (Section 4) | the model's uncertainty is meaningful early on |
| unknown molecules certainly present | open-set embedding + distance threshold (Section 5) | unknowns are far from known clusters |
| labels from a different source (simulation) | domain adaptation (Module 18) | source and target share classes |

---
## ✅ Checkpoint

1. Why might k-means on autoencoder codes group a ribosome with a proteasome?
   <details><summary>Answer</summary>The code must store everything needed to reconstruct the box, including orientation, position and overall size/blur; k-means may group by these nuisance factors rather than by molecular identity.</details>
2. In SimCLR, what decides what the features become invariant to?
   <details><summary>Answer</summary>The augmentations: whatever differs between two views of the same particle (rotation, shift, noise, contrast) is what the loss teaches the encoder to ignore.</details>
3. Write the prototype of class k and the ProtoNet class probability.
   <details><summary>Answer</summary>c_k = mean of the support embeddings f(x) of class k; p(y=k|x) = softmax over k of −d(f(x), c_k), with d the squared Euclidean distance.</details>
4. What does distribution calibration borrow from the base classes, and when could that hurt?
   <details><summary>Answer</summary>Their feature means and covariances (how classes vary); it hurts when the novel class does not resemble its nearest base classes, so the borrowed spread is wrong.</details>
5. HAL's score has two terms. What does each measure?
   <details><summary>Answer</summary>Entropy measures the classifier's uncertainty on a particle; the discriminator probability measures how unlike the labelled set (how under-represented) the particle's features are.</details>
6. At test time, how does Du *et al.*'s open-set method decide whether two subtomograms show the same molecule?
   <details><summary>Answer</summary>It embeds both and compares their L2 distance with a threshold chosen by cross-validation — no classification into known classes is needed, so it works for unseen classes.</details>

## 🧠 Questions & discussion

1. Contrastive learning treats every other particle in a batch as a "negative". What goes wrong if the batch contains
   many copies of the same molecule (e.g. a ribosome-rich cell)? How do MoCo or SupCon address related issues?
2. Few-shot benchmarks split classes into base and novel. In real cryo-ET, what would a fair base/novel split look like,
   and how could leakage sneak in (think homologous complexes)?
3. Active learning picks "informative" particles. Could it also pick particles that are simply *junk* (ice, carbon
   edge)? How would you guard against wasting expert time?
4. An open-set detector flags 30 % of a new tomogram as "unknown". List three biological and three technical reasons.
5. Which of this module's methods would you combine for a real project on a new cell type, and in which order?

## 🚀 Bonus — going further

- **MoCo.** Replace SimCLR's in-batch negatives by a queue of 128 past embeddings and a momentum encoder
  ($\theta_k \leftarrow 0.999\,\theta_k + 0.001\,\theta_q$), as in Gupta *et al.*
- **SupCon + calibration.** Add FSCC's supervised contrastive term ($\lambda = 0.05$, τ = 0.07) to `train_supervised` and
  re-run Section 3.3.
- **Task adaptation.** Implement a tiny version of ProtoNet-CE: a self-attention layer over the support prototypes,
  trained with the encoder frozen.
- **Gradient-norm acquisition.** Add an AL-GradNorm-style score (the norm of the loss gradient under the model's own
  predicted labels) to `query` and compare.
- **Official code.** Explore AITom's `few_shot/protonet/`, `active/hal/` and `openset/` folders; the repository's
  [`FewShotParticleDetection`](../../FewShotParticleDetection/) tutorial applies few-shot ideas to particle picking.

## 📝 Lab note

```text
Module 17 — date:
Input: 24³ subtomograms (15 Å voxels, SNR 0.1, ±60° wedge); 6 known + 5 novel zoo molecules
Output: AE/SimCLR features, linear-probe curves, few-shot accuracies, active-learning curves, open-set AUROC
Method: autoencoder + k-means; NT-Xent with z-rotations; ProtoNet episodes; distribution calibration; HAL scores;
        LMCL + center loss
Parameter I changed and what happened:
A failure mode / surprise:
Paper link: Zeng 2018; Liu S 2019; Gupta 2022; Li 2020; Gao 2022; Du 2021; Du 2019
```

## 📚 References

1. Zeng X, Leung MR, Zeev-Ben-Mordehai T, Xu M. A convolutional autoencoder approach for mining features in cellular electron cryo-tomograms and weakly supervised coarse segmentation. *J Struct Biol* 202(2):150–160 (2018). doi:10.1016/j.jsb.2017.12.015
2. Liu S, Du X, Xi R, Xu F, Zeng X, Zhou B, Xu M. Semi-supervised macromolecule structural classification in cellular electron cryo-tomograms using 3D autoencoding classifier. BMVC 2019. *(described from code only)*
3. Gupta T, He X, Uddin MR, Zeng X, Zhou A, Zhang J, Freyberg Z, Xu M. Self-supervised learning for macromolecular structure classification based on cryo-electron tomograms. *Front Physiol* 13:957484 (2022). doi:10.3389/fphys.2022.957484
4. Li R, Yu L, Zhou B, Zeng X, Wang Z, Yang X, Zhang J, Gao X, Jiang R, Xu M. Few-shot learning for classification of novel macromolecular structures in cryo-electron tomograms. *PLOS Comput Biol* 16(11):e1008227 (2020). doi:10.1371/journal.pcbi.1008227
5. Gao S, Zeng X, Xu M, Zhang F. FSCC: Few-shot learning for macromolecule classification based on contrastive learning and distribution calibration in cryo-electron tomography. *Front Mol Biosci* 9:931949 (2022). doi:10.3389/fmolb.2022.931949
6. Du X, Wang H, Zhu Z, Zeng X, Chang YW, Zhang J, Xing E, Xu M. Active learning to classify macromolecular structures in situ for less supervision in cryo-electron tomography. *Bioinformatics* (2021). doi:10.1093/bioinformatics/btab123
7. Du X, Zeng X, Zhou B, Singh A, Xu M. Open-set recognition of unseen macromolecules in cellular electron cryo-tomograms by soft large margin centralized cosine loss. BMVC 2019.
8. Wang T, Li B, Zhang J, Zeng X, Uddin MR, Wu W, Xu M. Deep active learning for cryo-electron tomography classification. IEEE ICIP 2022. *(content not verified)*
9. Zeng Y, Howe G, Zeng X, Zhang J, Chang YW, Xu M. Unsupervised domain alignment based open set structural recognition of macromolecules captured by cryo-electron tomography. IEEE ICIP 2021. *(content not verified)*
10. Wang T, Li X, Yang P, Hu G, Zeng X, Huang S, Xu C-Z, Xu M. Boosting active learning via improving test performance. AAAI 2022.
11. Li X, Yang P, Gu Y, Zhan X, Wang T, Xu M, Xu C. Deep active learning with noise stability. AAAI 2024, 38(12):13655–13663.
12. Chen T, Kornblith S, Norouzi M, Hinton G. A simple framework for contrastive learning of visual representations (SimCLR). ICML 2020. arXiv:2002.05709
13. Snell J, Swersky K, Zemel R. Prototypical networks for few-shot learning. NeurIPS 2017. arXiv:1703.05175
14. Yang S, Liu L, Xu M. Free lunch for few-shot learning: distribution calibration. ICLR 2021. arXiv:2101.06395
15. Wang H, Wang Y, Zhou Z, *et al.* CosFace: Large margin cosine loss for deep face recognition. CVPR 2018. arXiv:1801.09414
16. Wen Y, Zhang K, Li Z, Qiao Y. A discriminative feature learning approach for deep face recognition (center loss). ECCV 2016. doi:10.1007/978-3-319-46478-7_31

## ➡️ Next

**[18 · From simulation to reality: domain shift, adaptation and synthetic data](18_simulation_to_reality.ipynb)** —
every network so far was trained and tested on the same simulator; real tomograms are different. Previous:
[16 · Interpretability and segmentation](16_interpretability_segmentation.ipynb). Course overview: [`README.md`](../README.md).